# Phase 5: Advanced Feature Engineering

In this phase, we transform raw sensor data into intelligent signals that help the model detect patterns and anomalies more effectively.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Load data
df = pd.read_csv("../datasets/predictive_maintenance.csv")
encoder = LabelEncoder()
df["Type"] = encoder.fit_transform(df["Type"])

### Step 52 & 53: Rolling Mean and Standard Deviation

In [ ]:
df["temp_rolling_mean"] = df["Air temperature"].rolling(window=5).mean()
df["temp_rolling_std"] = df["Air temperature"].rolling(window=5).std()
print("Rolling features created.")

### Step 54 & 55: Lag and Difference Features

In [ ]:
df["temp_lag_1"] = df["Air temperature"].shift(1)
df["temp_lag_2"] = df["Air temperature"].shift(2)
df["temp_diff"] = df["Air temperature"].diff()
print("Lag and difference features created.")

### Step 56 & 57: RPM Stability and Torque Ratio

In [ ]:
df["rpm_stability"] = df["Rotational speed"].rolling(window=10).std()
df["torque_temp_ratio"] = df["Torque"] / df["Air temperature"]
print("Domain-inspired features created.")

### Step 63 & 64: Anomaly Flags

In [ ]:
threshold = df["Torque"].mean() + 2 * df["Torque"].std()
df["torque_anomaly"] = (df["Torque"] > threshold).astype(int)
print(f"Anomaly threshold set at: {threshold:.2f}")

### Step 58 & 59: Remove NaN and Retrain Model

In [ ]:
# Drop rows with NaN values created by rolling/lag
df = df.dropna()

# Split Features & Target
X = df.drop(["Machine failure", "UID", "Product ID"], axis=1)
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Retrain
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Model retrained with engineered features.")

### Step 60: Compare Results

In [ ]:
print("Classification Report:")
print(classification_report(y_test, predictions))

roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"ROC-AUC Score: {roc_auc:.4f}")

### Step 61: Feature Importance Again

In [ ]:
importance = model.feature_importances_
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print("Top 10 Important Features:")
print(feature_importance.head(10))

### Step 62: Visualize Rolling Mean

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df["temp_rolling_mean"].iloc[:500]) # Plotting first 500 for clarity
plt.title("Temperature Rolling Mean (First 500 records)")
plt.show()